In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = format_context_with_boundaries(df, lang="ae")

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.203764,1.273764,1.273764,1.353764,m,æ,m,masquerade,usctimit_ema_m1_081_085.mat_0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
1,1.273764,1.353764,1.353764,1.443764,æ,s,m_æ_.,masquerade,usctimit_ema_m1_081_085.mat_0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
2,1.353764,1.443764,1.443764,1.513764,s,k,m_æ_._s,masquerade,usctimit_ema_m1_081_085.mat_0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
3,1.443764,1.513764,1.513764,1.653764,k,ɝ,m_æ_._s_k,masquerade,usctimit_ema_m1_081_085.mat_0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
4,1.513764,1.653764,1.653764,1.763764,ɝ,eɪ,m_æ_._s_k_ɝ_.,masquerade,usctimit_ema_m1_081_085.mat_0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
43454,23.919070,23.999070,23.999070,24.089070,ʃ,ʌ,k_ɑ_m_._p_ʌ_._t_ɪ_._ʃ,competition,usctimit_ema_f1_121_125.mat_35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43455,23.999070,24.089070,24.089070,24.149070,ʌ,n,k_ɑ_m_._p_ʌ_._t_ɪ_._ʃ_ʌ,competition,usctimit_ema_f1_121_125.mat_35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43456,24.149070,24.179070,24.179070,24.209070,h,w,h,which,usctimit_ema_f1_121_125.mat_36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43457,24.179070,24.209070,24.209070,24.239070,w,ɪ,h_w,which,usctimit_ema_f1_121_125.mat_36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1


In [4]:
with open("../../pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

100%|██████████| 48576/48576 [00:00<00:00, 973307.75it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 48576/48576 [00:00<00:00, 164858.89it/s]


In [7]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [8]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone1_freq"] = df.apply(lambda x: monogram_counts.get(x["phone1"], None), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [9]:
pd.DataFrame(data).to_csv("surprisal_aeema.csv", index=False)